There are many ways to find the **second highest salary** in SQL. The best approach depends on the SQL database and whether you want to handle duplicate salaries.

Assume the table:

```sql
Employee
---------
id | name | salary
------------------
1  | Alice | 5000
2  | Bob   | 7000
3  | Carol | 6000
4  | David | 7000
5  | Eve   | 4000
```

> If the highest salary occurs multiple times, the **second highest salary** is **6000**.

### 1. Using `MAX()` with a subquery (Most common)

```sql
SELECT MAX(salary) AS second_highest_salary
FROM Employee
WHERE salary < (
    SELECT MAX(salary)
    FROM Employee
);
```

**How it works:**

* Find the maximum salary.
* Exclude it.
* Find the maximum among the remaining salaries.

---

### 2. Using `ORDER BY` and `LIMIT` (MySQL, PostgreSQL)

```sql
SELECT DISTINCT salary
FROM Employee
ORDER BY salary DESC
LIMIT 1 OFFSET 1;
```

or

```sql
SELECT DISTINCT salary
FROM Employee
ORDER BY salary DESC
LIMIT 1, 1;
```

(MySQL syntax)

---

### 3. Using `DENSE_RANK()` (Recommended)

```sql
SELECT salary
FROM (
    SELECT salary,
           DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
    FROM Employee
) t
WHERE rnk = 2;
```

**Why use `DENSE_RANK`?**

* Correctly handles duplicate salaries.
* Returns the second distinct highest salary.

---

### 4. Using `ROW_NUMBER()` (If duplicates are not a concern)

```sql
SELECT salary
FROM (
    SELECT salary,
           ROW_NUMBER() OVER (ORDER BY salary DESC) AS rn
    FROM Employee
) t
WHERE rn = 2;
```

**Note:** If the highest salary appears multiple times, this returns the second row, not the second distinct salary.

---

### 5. Using `RANK()`

```sql
SELECT salary
FROM (
    SELECT salary,
           RANK() OVER (ORDER BY salary DESC) AS rnk
    FROM Employee
) t
WHERE rnk = 2;
```

**Note:** If the highest salary has duplicates, there may be no row with `rnk = 2` because `RANK()` skips rank values.

Example:

* 7000 → Rank 1
* 7000 → Rank 1
* 6000 → Rank 3

---

### 6. Using a correlated subquery

```sql
SELECT DISTINCT e1.salary
FROM Employee e1
WHERE 1 = (
    SELECT COUNT(DISTINCT e2.salary)
    FROM Employee e2
    WHERE e2.salary > e1.salary
);
```

**Explanation:**

* Find salaries where exactly one distinct salary is higher.

---

### 7. Using `TOP` (SQL Server)

```sql
SELECT TOP 1 salary
FROM (
    SELECT DISTINCT TOP 2 salary
    FROM Employee
    ORDER BY salary DESC
) t
ORDER BY salary;
```

---

### 8. Using `FETCH FIRST` (Oracle, DB2, PostgreSQL)

```sql
SELECT DISTINCT salary
FROM Employee
ORDER BY salary DESC
OFFSET 1 ROW
FETCH FIRST 1 ROW ONLY;
```

---

## Comparison

| Method                      | Handles Duplicates      | Database Support          | Recommended |
| --------------------------- | ----------------------- | ------------------------- | ----------- |
| `MAX()` + Subquery          | ✅ Yes                   | All SQL databases         | ⭐⭐⭐⭐⭐       |
| `ORDER BY` + `LIMIT/OFFSET` | ✅ Yes (with `DISTINCT`) | MySQL, PostgreSQL         | ⭐⭐⭐⭐        |
| `DENSE_RANK()`              | ✅ Yes                   | Most modern SQL databases | ⭐⭐⭐⭐⭐       |
| `ROW_NUMBER()`              | ❌ No                    | Most modern SQL databases | ⭐⭐⭐         |
| `RANK()`                    | ⚠️ Can skip ranks       | Most modern SQL databases | ⭐⭐          |
| Correlated Subquery         | ✅ Yes                   | All SQL databases         | ⭐⭐⭐         |
| `TOP`                       | ✅ Yes                   | SQL Server                | ⭐⭐⭐⭐        |
| `FETCH FIRST`               | ✅ Yes                   | Oracle, DB2, PostgreSQL   | ⭐⭐⭐⭐        |

### Interview tip

For SQL interviews, these are the three most commonly expected solutions:

1. `MAX()` with a subquery (works across databases)
2. `DENSE_RANK()` (best for handling duplicates)
3. `ORDER BY` with `LIMIT/OFFSET` (simple for MySQL/PostgreSQL)


select count(*) from table group by co1,col2 having count(*) >>1

In [0]:
%sql
select * from salary where salary not in (select max)

In [0]:
| emp_id | emp_name   | manager_id | manager_name | salary |
|--------|------------|------------|--------------|--------|
| 1      | Alice      | 3          | Charlie      | 5000   |
| 2      | Bob        | 3          | Charlie      | 4800   |
| 3      | Charlie    | 3          | Charlie      | 7000   |
| 4      | David      | 3          | Charlie      | 4700   |

In [0]:
%sql
select EMPLOYEE_ID,DEPARTMENT_ID,max(SALARY) from my_learning.my_raw_feed.employees group by EMPLOYEE_ID,DEPARTMENT_ID

In [0]:
%sql
select max(salary) from my_learning.my_raw_feed.employees where salary not in (select max(salary) from my_learning.my_raw_feed.employees)

In [0]:
%sql
select * from (select EMPLOYEE_ID, DEPARTMENT_ID,salary ,row_number() over(order by salary desc) as rn from my_learning.my_raw_feed.employees ) t where rn=2

In [0]:
%sql
select distinct EMPLOYEE_ID,salary from my_learning.my_raw_feed.employees order by salary desc limit 1 offset 1;

In [0]:
%sql
SELECT DISTINCT e1.salary
FROM my_learning.my_raw_feed.employees e1
WHERE 1 = (
    SELECT COUNT(DISTINCT e2.salary)
    FROM my_learning.my_raw_feed.employees e2
    WHERE e2.salary > e1.salary
);

## Running Total

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

df = spark.table("my_learning.my_raw_feed.employees")

window_spec = Window.partitionBy("DEPARTMENT_ID").orderBy("SALARY").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_running_total = df.withColumn(
    "running_total_salary",
    F.sum("SALARY").over(window_spec)
)

display(df_running_total)